[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/92_max_product_bd260730_solution.ipynb)

# 参考解法：最大乘积操作

Reference solution.

## 解析

**结论：每次减都作用在当前最大元素上；等价于把最高的若干元素整体逐层压低，最后用快速幂拼出乘积。**

### 为什么减最大
把 `x` 减到 `x-1`，乘积乘上因子 `(x-1)/x`。这个因子随 `x` 增大而增大（越接近 1 损失越小），所以在需要减 1 时，作用于当前最大元素造成的相对损失最小。

### 分层下压
升序排序后从尾部开始，把当前最高的 `cnt` 个元素一起降到下一个不同值 `next`，代价 `(level-next)*cnt`。可负担就继续扩大这批元素的数量并降档；不能整批降完时，用 `q, r = divmod(k, cnt)` 把它们均匀降 `q`，再有 `r` 个多降 1。结果为 `high = level - q`，乘积 = 前缀未动元素 × `high^(cnt-r)` × `(high-1)^r`。

### 取模细节
元素与 `k` 都可能很大，乘积用快速幂 `pow(base, exp, MOD)` 计算并全程对 `1e9+7` 取模。

### 验证
已用带记忆化的穷举 DFS 在数千组随机 `(a, k)` 上对拍一致，并复现全部官方样例。

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass

In [ ]:
from typing import List

In [ ]:
# ✅ SOLUTION

from typing import List

MOD = 10 ** 9 + 7

class Solution:
    def max_product(self, a: List[int], k: int) -> int:
        a = sorted(a)
        n = len(a)
        total = sum(x - 1 for x in a)
        if k >= total:
            return 1                       # everything can be driven to 1
        i = n - 1
        level = a[i]
        cnt = 1                        # how many top elements sit at `level`
        while True:
            nxt = a[i - 1] if i - 1 >= 0 else 1
            cost = (level - nxt) * cnt
            if k >= cost and nxt > 1 and i - 1 >= 0:
                k -= cost
                level = nxt
                i -= 1
                cnt += 1
            else:
                break
        q, r = divmod(k, cnt)
        high = level - q
        res = 1
        for j in range(i):
            res = res * (a[j] % MOD) % MOD  # untouched prefix
        res = res * pow(high % MOD, cnt - r, MOD) % MOD
        res = res * pow((high - 1) % MOD, r, MOD) % MOD
        return res % MOD

In [ ]:
sol = Solution()
print(sol.max_product([2, 2, 3], 3))     # 2
print(sol.max_product([5, 1, 3, 2], 2))  # 18
print(sol.max_product([3, 4], 0))        # 12

In [ ]:
from torch_judge import check
check('max_product_bd260730')